# Seedream Image Generation Test

This notebook runs on your local Windows Python environment and tests the same Ark Seedream image generation path used by the app.

Before running:
- Select a local Windows Python kernel, not the container kernel.
- Make sure `.env` contains `ARK_API_KEY`.
- If needed, run the install cell first.

In [ ]:
%pip install openai python-dotenv ipykernel

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Image, display

repo_root = Path.cwd()
env_file = repo_root / '.env'
if env_file.exists():
    load_dotenv(env_file)
    print(f'Loaded env from: {env_file}')
else:
    print('No .env found in current working directory; using existing process environment values.')

ARK_API_KEY = os.getenv('ARK_API_KEY')
ARK_IMAGE_BASE_URL = os.getenv('ARK_IMAGE_BASE_URL', 'https://ark.cn-beijing.volces.com/api/v3')
ARK_IMAGE_MODEL_ID = os.getenv('ARK_IMAGE_MODEL_ID', 'doubao-seedream-5-0-lite-260128')
ARK_IMAGE_SIZE = os.getenv('ARK_IMAGE_SIZE', '2K')

print('ARK_IMAGE_BASE_URL =', ARK_IMAGE_BASE_URL)
print('ARK_IMAGE_MODEL_ID =', ARK_IMAGE_MODEL_ID)
print('ARK_IMAGE_SIZE =', ARK_IMAGE_SIZE)
print('ARK_API_KEY configured =', bool(ARK_API_KEY))

if not ARK_API_KEY:
    raise RuntimeError('ARK_API_KEY is not set. Put it in .env or the current Windows environment before running this notebook.')

In [ ]:
prompt = '一间带大落地窗的现代建筑工作室，阳光斜照进来，木质桌面上摆着建筑模型，画面干净，写实，高细节'
input_image_url = None

client = OpenAI(api_key=ARK_API_KEY, base_url=ARK_IMAGE_BASE_URL)

def generate_seedream_image(prompt: str, image: str | None = None, size: str = ARK_IMAGE_SIZE):
    extra_body = {
        'sequential_image_generation': 'disabled',
        'watermark': False,
    }
    if image:
        extra_body['image'] = image

    response = client.images.generate(
        model=ARK_IMAGE_MODEL_ID,
        prompt=prompt,
        size=size,
        response_format='url',
        extra_body=extra_body,
    )

    data = getattr(response, 'data', None) or []
    if not data:
        raise RuntimeError(f'No image data returned: {response}')

    first = data[0]
    image_url = getattr(first, 'url', None)
    if not image_url:
        raw = response.model_dump() if hasattr(response, 'model_dump') else str(response)
        raise RuntimeError(f'Image URL missing in response: {raw}')

    return response, image_url

In [ ]:
response, image_url = generate_seedream_image(prompt=prompt, image=input_image_url)

raw = response.model_dump() if hasattr(response, 'model_dump') else str(response)
print('Generated image URL:', image_url)
print(raw)

display(Image(url=image_url))

## How To Use

- For text-to-image, keep `input_image_url = None`.
- For image-to-image, replace `input_image_url` with an accessible image URL and rerun the last two code cells.
- If this notebook succeeds but the app still fails, the remaining issue is in the app runtime path rather than Ark account access.